In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
nltk.data.path.append("C:/nltk_data")

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [5]:
from sklearn.model_selection import train_test_split

X = data['text']
y = data ['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

## Data Preprocessing

In [6]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Anast\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [11]:
import re 
from bs4 import BeautifulSoup
import html

In [12]:
def clean_html(text):
    if not isinstance(text, str):
        return " "
    
    #removing <script> & <style> blocks
    text = re.sub(r'<(script|style).*?>.*?</\1>', " ", text, flags=re.DOTALL | re.IGNORECASE)

    #removing HTML comments
    text = re.sub(r'<!--.*?-->', " ", text, flags=re.DOTALL)

    soup = BeautifulSoup(text, 'html.parser')
    text = soup.get_text(separator=" ")

    text = html.unescape(text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [13]:
data['clean_text'] = data['text'].apply(clean_html)

C:\Users\Anast\AppData\Local\Temp\ipykernel_22484\2357385022.py:11: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(text, 'html.parser')


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [14]:
def normalize_text(text):
    if not isinstance(text, str):
        return " "
    
    #remove special characters 
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    #remove numbers 
    text = re.sub(r"\d+", " ", text)

    #remove single characters 
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)

    #remove single characters from the start 
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text)

    #remove multiple spaces with single space
    text = re.sub(r"\s+", " ", text).strip()

    #remove prefixed 'b'
    text = re.sub(r"^b\s+", " ", text)

    #convert to lowercase
    text = text.lower()

    return text 

In [15]:
data['normalized_text'] = data['clean_text'].apply(normalize_text)
data

,text,label,clean_text,normalized_text
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...
1,Will do.,0,Will do.,will do
2,Nora--Cheryl has emailed dozens of memos about...,0,Nora--Cheryl has emailed dozens of memos about...,nora cheryl has emailed dozens of memos about ...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam know that this proposal might ...
4,fyi,0,fyi,fyi
...,...,...,...,...
995,So what's the latest? It sounds contradictory ...,0,So what's the latest? It sounds contradictory ...,so what the latest it sounds contradictory and...
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",transfer of million pounds to youraccount my n...
997,Barb I will call to explain. Are you back in t...,0,Barb I will call to explain. Are you back in t...,barb will call to explain are you back in the ...
998,Yang on travelNot free tonite.May work tomorrow,0,Yang on travelNot free tonite.May work tomorrow,yang on travelnot free tonite may work tomorrow


## Now let's work on removing stopwords
Remove the stopwords.

In [16]:
stop_words = set(stopwords.words('english'))

In [17]:
def remove_stop_wo(text):
    if not isinstance(text, str):
        return ""
    
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]

    return ' '.join(filtered_words)

In [18]:
data['wo_stop_words'] = data['normalized_text'].apply(remove_stop_wo)
data

,text,label,clean_text,normalized_text,wo_stop_words
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...
1,Will do.,0,Will do.,will do,
2,Nora--Cheryl has emailed dozens of memos about...,0,Nora--Cheryl has emailed dozens of memos about...,nora cheryl has emailed dozens of memos about ...,nora cheryl emailed dozens memos haiti weekend...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam know that this proposal might ...,dear sir fmadam know proposal might surprise e...
4,fyi,0,fyi,fyi,fyi
...,...,...,...,...,...
995,So what's the latest? It sounds contradictory ...,0,So what's the latest? It sounds contradictory ...,so what the latest it sounds contradictory and...,latest sounds contradictory af decide shall ta...
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",transfer of million pounds to youraccount my n...,transfer million pounds youraccount name mr ej...
997,Barb I will call to explain. Are you back in t...,0,Barb I will call to explain. Are you back in t...,barb will call to explain are you back in the ...,barb call explain back country
998,Yang on travelNot free tonite.May work tomorrow,0,Yang on travelNot free tonite.May work tomorrow,yang on travelnot free tonite may work tomorrow,yang travelnot free tonite may work tomorrow


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [ ]:
nltk.download('punkt', download_dir='C:/nltk_data')
nltk.download('wordnet', download_dir='C:/nltk_data')
nltk.download('omw-1.4', download_dir='C:/nltk_data')

[nltk_data] Downloading package punkt to C:/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [22]:
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer 

lemmatizer = WordNetLemmatizer

def tokanize(text):
    if not isinstance(text, str):
        return ""
    
    words = word_tokenize(text)

    cleaned_words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(cleaned_words)

In [23]:
data['clean_text'] = data['wo_stop_words'].apply(tokanize)
data

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\Anast/nltk_data'
    - 'c:\\Users\\Anast\\AppData\\Local\\Programs\\Python\\Python313\\nltk_data'
    - 'c:\\Users\\Anast\\AppData\\Local\\Programs\\Python\\Python313\\share\\nltk_data'
    - 'c:\\Users\\Anast\\AppData\\Local\\Programs\\Python\\Python313\\lib\\nltk_data'
    - 'C:\\Users\\Anast\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - 'C:/nltk_data'
**********************************************************************


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [24]:
ham_messages = data[data['label'] == 0]['wo_stop_words']
spam_messages = data[data['label'] == 1]['wo_stop_words']

ham_words = " ".join(ham_messages).split()
spam_words = " ".join(spam_messages).split()

ham_freq = pd.Series(ham_words).value_counts().head(10)
spam_freq = pd.Series(spam_words).value_counts().head(10)

ham_freq
spam_freq

money          987
account        840
bank           781
us             737
transaction    544
business       514
country        480
fund           475
million        442
transfer       425
Name: count, dtype: int64

## Extra features

In [27]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€","$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data['money_mark'] = data['wo_stop_words'].str.contains(money_simbol_list)*1
data['suspicious_words'] = data['wo_stop_words'].str.contains(suspicious_words)*1
data['text_len'] = data['wo_stop_words'].apply(lambda x: len(x)) 

""" 
X_test['money_mark'] = X_test['preprocessed_text'].str.contains(money_simbol_list)*1
X_test['suspicious_words'] = X_test['preprocessed_text'].str.contains(suspicious_words)*1
X_test['text_len'] = X_test['preprocessed_text'].apply(lambda x: len(x))  """

data.head()

,text,label,clean_text,normalized_text,wo_stop_words,money_mark,suspicious_words,text_len
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...,1,1,1515
1,Will do.,0,Will do.,will do,,1,0,0
2,Nora--Cheryl has emailed dozens of memos about...,0,Nora--Cheryl has emailed dozens of memos about...,nora cheryl has emailed dozens of memos about ...,nora cheryl emailed dozens memos haiti weekend...,1,0,113
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam know that this proposal might ...,dear sir fmadam know proposal might surprise e...,1,1,1358
4,fyi,0,fyi,fyi,fyi,1,0,3


## How would you create a Bag of Words with the CountVectorizer method?

## TD-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
text = data['wo_stop_words'].tolist()

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(text)

X_bow.shape

(1000, 30726)

### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to find the **best feature representation**.

You can work with teams of two persons (recommended).

In [ ]:
# Your code